# SemEval-2026 Task 2 — Notebook chạy trên Colab

Notebook này KHÔNG chứa logic xử lý — mọi logic nằm trong thư mục `src/`.
Notebook chỉ có vai trò: (1) cài đặt môi trường Colab, (2) gọi hàm từ `src/` để chạy.

**Thứ tự chạy:** clone repo -> tải data -> kiểm tra token length -> chạy baseline (Trục 1) -> train model chính (V1, V3) -> ensemble -> so sánh loss strategy (Trục 2) -> tổng hợp bảng + vẽ biểu đồ.

## 0. Cài đặt môi trường

In [ ]:
!git clone https://github.com/Ciaranguyen/SemEval-2026-task2-lamanhnguyen.git
%cd SemEval-2026-task2-lamanhnguyen
!pip install -r requirements.txt -q

## 1. Tải dữ liệu
Upload các file .csv (train_subtask1.csv, train_subtask2a.csv, ...) vào `data/raw/`.
Có thể mount Google Drive để tránh upload lại mỗi lần Colab reset.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Đổi đường dẫn này đúng theo nơi em lưu data trên Drive
!cp /content/drive/MyDrive/SemEval2026_data/*.csv data/raw/

## 2. Kiểm tra độ dài token thật (bảo vệ lựa chọn max_len=128)

In [ ]:
!python -m src.check_token_length

## 3. Trục 1 — Chạy các baseline (lexicon, TF-IDF, BERT official frozen)

In [ ]:
from src.utils import load_config
from src.data import load_and_merge_train_data, split_train_val
from src.baselines import LexiconBaseline, TfidfRidgeBaseline, OfficialBertFrozenBaseline
from src.evaluate import evaluate_va, build_comparison_table
import numpy as np

cfg = load_config('configs/config.yaml')
df = load_and_merge_train_data(cfg['data']['raw_dir'], cfg['data']['train_files'], cfg['data']['columns'])
df_train, df_val = split_train_val(df, cfg['data']['val_split'], cfg['data']['seed'])
y_true = df_val[['valence', 'arousal']].values

results = []

# Bậc 1: Lexicon (cần tải NRC-VAD-Lexicon.txt vào data/raw/ trước)
lex = LexiconBaseline(cfg['baselines']['lexicon_path'])
lex.calibrate(df_train['text'], df_train['valence'], df_train['arousal'])
pred_lex = lex.predict(df_val['text'])
results.append(evaluate_va(y_true, pred_lex, 'Lexicon (NRC-VAD)', 'Subtask 1'))

# Bậc 2: TF-IDF + Ridge
tfidf = TfidfRidgeBaseline(cfg['baselines']['tfidf_max_features'], cfg['baselines']['ridge_alpha'])
tfidf.fit(df_train['text'], df_train['valence'], df_train['arousal'])
pred_tfidf = tfidf.predict(df_val['text'])
results.append(evaluate_va(y_true, pred_tfidf, 'TF-IDF + Ridge', 'Subtask 1'))

# Bậc 3: Official linear(BERT) - baseline chính thức BTC
official = OfficialBertFrozenBaseline(cfg['baselines']['official_bert_frozen'])
official.fit(df_train['text'], df_train['valence'], df_train['arousal'])
pred_official = official.predict(df_val['text'])
results.append(evaluate_va(y_true, pred_official, 'linear(BERT) - Official', 'Subtask 1'))

build_comparison_table(results)

## 4. Trục 1 (tiếp) — Train BERT/DeBERTa single-task vs multi-task

In [ ]:
!python -m src.train --variant bert_singletask --target valence
!python -m src.train --variant deberta_singletask --target valence
!python -m src.train --variant v1
!python -m src.train --variant v3

## 5. Trục 2 — So sánh chiến lược Multi-task Loss

In [ ]:
!python -m src.train --variant v1 --loss_strategy fixed_weight
!python -m src.train --variant v1 --loss_strategy uncertainty_weighting

## 6. Tổng hợp bảng kết quả cuối cùng + vẽ biểu đồ
(Điền số liệu từ các bước trên vào bảng tổng hợp, dùng cho mục V.5.5 của báo cáo)

In [ ]:
# TODO: gộp toàn bộ kết quả các bước trên bằng build_comparison_table(),
# rồi dùng matplotlib/seaborn để vẽ bar chart so sánh (xem hướng dẫn trong báo cáo)